# Visualization tools for AMB3R

### Data Loading

In [ ]:
import numpy as np
import os
import open3d as o3d

NPZ_PATH          = 'path_to_your_results.npz'
SAVE_PATH         = 'path_to_save_visualization'
DOWNSAMPLE_FACTOR = 1

os.makedirs(SAVE_PATH, exist_ok=True)

data      = np.load(NPZ_PATH)
poses     = data['pose']                                         # (N, 4, 4)
pts       = data['pts']                                          # (N, H, W, 3)
conf      = data['conf']                                         # (N, H, W)
images    = (data['images'].transpose(0, 2, 3, 1) + 1.0) / 2.0  # (N, H, W, 3)
sky_masks = data['sky_mask']                                     # (N, H, W)
kf_idx    = data['kf_idx']                                       # (K,)
del data

if DOWNSAMPLE_FACTOR > 1:
    pts       = pts      [:, ::DOWNSAMPLE_FACTOR, ::DOWNSAMPLE_FACTOR, :]
    conf      = conf     [:, ::DOWNSAMPLE_FACTOR, ::DOWNSAMPLE_FACTOR]
    images    = images   [:, ::DOWNSAMPLE_FACTOR, ::DOWNSAMPLE_FACTOR, :]
    sky_masks = sky_masks[:, ::DOWNSAMPLE_FACTOR, ::DOWNSAMPLE_FACTOR]

### Point Cloud Filtering and Visualization

In [ ]:
from amb3r.tools.pts_vis import get_pts_edge_mask

CONF_THRESHOLD        = 1e-1
EDGE_NORMAL_THRESHOLD = 5.0
EDGE_DEPTH_THRESHOLD  = 0.008

mask = get_pts_edge_mask(
    pts,
    sky_mask=sky_masks,
    conf=conf,
    conf_threshold=CONF_THRESHOLD,
    edge_normal_threshold=EDGE_NORMAL_THRESHOLD,
    edge_depth_threshold=EDGE_DEPTH_THRESHOLD,
)

total_pts    = mask.size
total_kf_pts = mask[kf_idx].size
valid_pts    = mask.reshape(-1).sum()
valid_kf_pts = mask[kf_idx].reshape(-1).sum()

print(f'All points : {valid_pts:>12,} / {total_pts:>12,}  ({100*(1 - valid_pts/total_pts):.1f}% pruned)')
print(f'KF  points : {valid_kf_pts:>12,} / {total_kf_pts:>12,}  ({100*(1 - valid_kf_pts/total_kf_pts):.1f}% pruned)')

In [ ]:
from amb3r.tools.vis import create_camera_frustum_pcd, draw_scene

CAM_WIDTH      = 0.015
CAM_HEIGHT     = 0.01
FOCAL_LENGTH   = 0.015
KEYFRAME_COLOR = [0.8, 0.2, 0.2]
MAX_KF_POINTS  = 10_000_000
POINT_SIZE     = 0.1

# --- Keyframe point cloud ---
pts_kf    = pts[kf_idx].reshape(-1, 3)[mask[kf_idx].reshape(-1)]
colors_kf = images[kf_idx].reshape(-1, 3)[mask[kf_idx].reshape(-1)]
print(f'Keyframe points before sampling: {pts_kf.shape[0]:,}')

if pts_kf.shape[0] > MAX_KF_POINTS:
    idx = np.random.choice(pts_kf.shape[0], size=MAX_KF_POINTS, replace=False)
    pts_kf, colors_kf = pts_kf[idx], colors_kf[idx]
print(f'Keyframe points after sampling : {pts_kf.shape[0]:,}')

pcd_kf = o3d.geometry.PointCloud()
pcd_kf.points = o3d.utility.Vector3dVector(pts_kf)
pcd_kf.colors = o3d.utility.Vector3dVector(colors_kf)
o3d.io.write_point_cloud(os.path.join(SAVE_PATH, 'sfm_kf_points_cleaned.ply'), pcd_kf)

# --- Camera frustums ---
camera_frustums = [
    create_camera_frustum_pcd(CAM_WIDTH, CAM_HEIGHT, FOCAL_LENGTH, p, KEYFRAME_COLOR)
    for p in poses
]
cam_pts    = np.vstack([np.asarray(f.points) for f in camera_frustums])
cam_colors = np.vstack([np.asarray(f.colors) for f in camera_frustums])
pcd_cam = o3d.geometry.PointCloud()
pcd_cam.points = o3d.utility.Vector3dVector(cam_pts)
pcd_cam.colors = o3d.utility.Vector3dVector(cam_colors)
o3d.io.write_point_cloud(os.path.join(SAVE_PATH, 'sfm_cameras.ply'), pcd_cam)

draw_scene([pcd_kf, *camera_frustums], point_size=POINT_SIZE)

### Scene Visualization and Rendering

In [ ]:
from amb3r.tools.vis import (
    create_sphere_pcd_template,
    build_interactive_scene,
    collect_manual_camera_path,
    convert_pinhole_params_to_poses,
    smooth_trajectory_robust,
    interpolate_smooth_trajectory,
    render_offline_video,
)

# CAM_WIDTH, CAM_HEIGHT, FOCAL_LENGTH, POINT_SIZE inherited from cell 3
NUM_VIDEO_FRAMES    = 1000
KEYFRAME_CAM_COLOR  = [0.8, 0.2, 0.2]
SELECTED_CAM_COLOR  = [0.8, 0.8, 0.2]
SPHERE_RADIUS       = 0.001
SPHERE_RESOLUTION   = 5
INTERACTIVE_MAX_PTS = 2_000_000
SMOOTH_S_FACTOR     = 0.5
SMOOTH_ROT_WINDOW   = 3
USE_KF_PCD          = True

camera_path_file = os.path.join(SAVE_PATH, 'manual_camera_path.npy')
sphere_template  = create_sphere_pcd_template(radius=SPHERE_RADIUS, resolution=SPHERE_RESOLUTION)

pcd_interactive, frustums_interactive = build_interactive_scene(
    pts_all=pts, image_all=images, poses_all=poses, kf_idx_all=kf_idx, mask=mask,
    cam_width=CAM_WIDTH, cam_height=CAM_HEIGHT, focal_length=FOCAL_LENGTH,
    frustum_color=KEYFRAME_CAM_COLOR, frustum_sphere_template=sphere_template,
    points_per_line=10, max_points=INTERACTIVE_MAX_PTS,
    pcd_scene=pcd_kf if USE_KF_PCD else None,
)

initial_poses = None
if os.path.exists(camera_path_file):
    initial_poses = np.load(camera_path_file)
    print(f'Loaded {len(initial_poses)} existing camera keyframes from {camera_path_file}')

manual_params, is_loop = collect_manual_camera_path(
    pcd_interactive, frustums_interactive,
    cam_width=CAM_WIDTH, cam_height=CAM_HEIGHT, focal_length=FOCAL_LENGTH,
    frustum_sphere_template=sphere_template, selected_cam_color=SELECTED_CAM_COLOR,
    num_video_frames=NUM_VIDEO_FRAMES, preview_s_factor=SMOOTH_S_FACTOR,
    preview_rot_window=SMOOTH_ROT_WINDOW, initial_poses=initial_poses,
    point_size=POINT_SIZE,
)

if len(manual_params) < 2:
    raise ValueError('Need at least 2 camera keyframes.')
print(f'Collected {len(manual_params)} manual keyframes.')

manual_poses = convert_pinhole_params_to_poses(manual_params)
np.save(camera_path_file, manual_poses)
print(f'Saved {len(manual_poses)} cameras to {camera_path_file}')

if is_loop:
    manual_poses = np.concatenate([manual_poses, manual_poses[:1]])

poses_smooth = smooth_trajectory_robust(
    manual_poses, translation_smoothing_factor=SMOOTH_S_FACTOR, rotation_filter_window=SMOOTH_ROT_WINDOW,
)
interpolated_poses = interpolate_smooth_trajectory(poses_smooth, num_output_frames=NUM_VIDEO_FRAMES)

render_offline_video(
    pts_all=pts, image_all=images, poses_all=poses, kf_idx_all=kf_idx, mask=mask,
    interpolated_poses=interpolated_poses, output_dir=SAVE_PATH,
    cam_width=CAM_WIDTH, cam_height=CAM_HEIGHT, focal_length=FOCAL_LENGTH,
    frustum_color=KEYFRAME_CAM_COLOR, frustum_sphere_template=sphere_template,
    save_video=True,
    pcd_scene=pcd_kf if USE_KF_PCD else None,
    point_size=POINT_SIZE,
)